# Этап 2. Подготовка датасета

### Автор: Балицкая Ксения

In [147]:
import json
import pandas as pd
from pathlib import Path
import numpy as np
import ast
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MultiLabelBinarizer

In [87]:
# Путь до наших сгенерированных данных в папке Fashion_CV/create_dataset/
DATA_PATH = Path('../create_dataset/data/dataset_annotations.json')

In [88]:
if not DATA_PATH.exists():
    print(f"Файл не найден по пути: {DATA_PATH.resolve()}")
else:
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    print(f"Успешно загружено {len(raw_data)} записей")

Успешно загружено 15000 записей


***

#### 1. Преобразование JSON в DataFrame

In [89]:
df_raw = pd.DataFrame.from_dict(raw_data, orient='index')
df_raw.head(2)

,plaid_shirt_dress,grey_sweater,neon_beanie,black_trousers,black_boots,ankle_boot_socks,black_jacket,neon_yellow_beanie,brown_bag,black_leather_jacket,...,blue_rings,brown_handbag,grey_and_white_striped_socks,black_ankle_strap_socks,white_pants,black_high_waisted_jean_pants,white_collar_shirt,chunky_scarf,scuffed_ankle_boots,brown belt
images_for_training/Paisley_Rose_Print_Dress_img_00000024.jpg,"{'item_type': 'shirt dress', 'demographic': 'w...","{'item_type': 'sweater', 'demographic': 'unise...","{'item_type': 'beanie', 'demographic': 'unisex...","{'item_type': 'trousers', 'demographic': 'unis...","{'item_type': 'boots', 'demographic': 'unisex'...","{'item_type': 'socks', 'demographic': 'unisex'...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
images_for_training/WOMEN1_Tees_Tanks_id_00004889_06_4_full.jpg,"{'item_type': 'shirt dress', 'demographic': 'w...","{'item_type': 'sweater', 'demographic': 'unise...","{'item_type': 'beanie', 'demographic': 'unisex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
df_stacked = df_raw.stack().reset_index()
df_stacked.columns = ['image_path', 'item_name', 'attributes']
df_attributes = pd.json_normalize(df_stacked['attributes'])

In [91]:
df = pd.concat([df_stacked[['image_path', 'item_name']], df_attributes], axis=1)
df.head(2)

,image_path,item_name,item_type,demographic,age_group,color,silhouette,design_features,length,volume,functional_elements,decorative_elements,accessories,season,purpose,style,usage_conditions
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,shirt dress,women,young adults 18-35,black with white grid,fitted,"button-down, V-neck",knee-length,tight,"[buttons, pockets]",[grid pattern],"[scarf, ankle boots]",fall,casual,grunge,"outdoor, casual walk"
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,sweater,unisex,all ages,heather grey,oversized,ribbed knit,None,loose,[pockets],[chunky knit],"[neon beanie, sunglasses]",winter,casual,minimalist,outdoor


In [92]:
# Размер датасета
df.shape

(62979, 17)

***

#### 2. Очистка данных

In [93]:
df.describe()

,image_path,item_name,item_type,demographic,age_group,color,silhouette,design_features,length,volume,functional_elements,decorative_elements,accessories,season,purpose,style,usage_conditions
count,62979,62979,62979,62979,62979,62979,54335,60226,45377,53968,61717,61066,60691,62949,62971,62966,62971
unique,15000,363,76,3,3,65,69,1219,26,5,493,405,460,7,27,44,38
top,images_for_training/Socialite_Halter_Dress_img...,plaid_shirt_dress,sweater,unisex,all ages,neon yellow,fitted,[ribbed knit],null,loose,[],[],[],all-season,casual,minimalist,outdoor
freq,18,14691,14954,41624,44838,14940,20493,14750,23841,27149,19125,20044,29195,26793,55697,24811,38522


In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62979 entries, 0 to 62978
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   image_path           62979 non-null  object
 1   item_name            62979 non-null  object
 2   item_type            62979 non-null  object
 3   demographic          62979 non-null  object
 4   age_group            62979 non-null  object
 5   color                62979 non-null  object
 6   silhouette           54335 non-null  object
 7   design_features      60226 non-null  object
 8   length               45377 non-null  object
 9   volume               53968 non-null  object
 10  functional_elements  61717 non-null  object
 11  decorative_elements  61066 non-null  object
 12  accessories          60691 non-null  object
 13  season               62949 non-null  object
 14  purpose              62971 non-null  object
 15  style                62966 non-null  object
 16  usag

Как мы видим, есть пропущенные значения, которые достаточно часто встречаются в некоторых колонках

In [95]:
for name in df.columns:
    if df[name].isna().sum() != 0:
        print(f"В колонке {name} - {df[name].isna().sum()} пропущенных значений")

В колонке silhouette - 8644 пропущенных значений
В колонке design_features - 2753 пропущенных значений
В колонке length - 17602 пропущенных значений
В колонке volume - 9011 пропущенных значений
В колонке functional_elements - 1262 пропущенных значений
В колонке decorative_elements - 1913 пропущенных значений
В колонке accessories - 2288 пропущенных значений
В колонке season - 30 пропущенных значений
В колонке purpose - 8 пропущенных значений
В колонке style - 13 пропущенных значений
В колонке usage_conditions - 8 пропущенных значений


Распишу стратегию работы с фотографиями:
1. У нас есть колонка `image_path`, которая не несет в себе никакой информации, помимо пути к нашим фотографиям. Эту колонку можно было бы удалить, но возможно, что на 3-м этапе она понадобится, когда будет обучение нейронной сети, поэтому эту колонку можно просто проигнорировать при подаче признаков в сеть
2. Также у нас есть проблемные колонки в которых отстуствует большая часть информации:
    * `silhouette` - силуэт (8644 пропуска)
    * `design_features` - детали дизайна (2753 пропуска)
    * `length` - длина (17602 пропуска)
    * `volume` - объем (9011 пропусков)
    * `functional_elements`/`decorative_elements` - карманы, молнии, рюши, принты (вместе 3175 пропусков)
    * `accessories` - наличие ремня, очков, сумки (2288 пропусков) \

    Для базового подбора одежды данные признаки можно удалить, а сотавить только `design_features`, заполнив NaN пустым списком [] т.к. это наиболее важная колонка
3. Остальные пропуски можно заполнить словом 'unknown' т.к. мы работаем не просто с данными, они должны соотвествовать фотографиям.

In [96]:
# Удаление колонок 
columns_to_drop = ["silhouette", "length", "volume", "functional_elements", "decorative_elements", "accessories"]

df = df.drop(columns=columns_to_drop)
df.shape

(62979, 11)

In [97]:
# Работа с колонкой design_features
df['design_features'].nunique

<bound method IndexOpsMixin.nunique of 0           button-down, V-neck
1                   ribbed knit
2                          None
3                      slim fit
4                          None
                  ...          
62974               ribbed knit
62975     [button-down, V-neck]
62976    [lacing, treaded sole]
62977             [ribbed knit]
62978             [ribbed knit]
Name: design_features, Length: 62979, dtype: object>

В этой колонке у нас есть не только пустые значения, но и нормальный список, или строка через запятую, или строка, которая выглядит, как список. Все это нужно привести к единому виду

In [98]:
def clean_list_column(x):
    # 1. Если это пустота 
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    
    # 2. Если это уже нормальный питоновский список
    if isinstance(x, list):
        return x
    
    # 3. Если это строка, которая выглядит как список 
    if isinstance(x, str) and x.startswith('['):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
            
    # 4. Если это просто строка через запятую 
    if isinstance(x, str):
        return [item.strip() for item in x.split(',') if item.strip()]
        
    return []

In [99]:
df['design_features'] = df['design_features'].apply(clean_list_column)

In [100]:
# Остальные пропуски заполняем словом 'unknown'
df = df.fillna('unknown')
df.head()

,image_path,item_name,item_type,demographic,age_group,color,design_features,season,purpose,style,usage_conditions
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,shirt dress,women,young adults 18-35,black with white grid,"[button-down, V-neck]",fall,casual,grunge,"outdoor, casual walk"
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,sweater,unisex,all ages,heather grey,[ribbed knit],winter,casual,minimalist,outdoor
2,images_for_training/Paisley_Rose_Print_Dress_i...,neon_beanie,beanie,unisex,all ages,neon yellow,[],unknown,casual,street style,unknown
3,images_for_training/Paisley_Rose_Print_Dress_i...,black_trousers,trousers,unisex,all ages,black,[slim fit],all-season,casual,minimalist,"outdoor, casual walk"
4,images_for_training/Paisley_Rose_Print_Dress_i...,black_boots,boots,unisex,all ages,black,[],winter,casual,grunge,outdoor


Для колонки `usage_conditions` тоже сделаем список в значениях, применив функцию выше

In [101]:
df['usage_conditions'] = df['usage_conditions'].apply(clean_list_column)
df.head()

,image_path,item_name,item_type,demographic,age_group,color,design_features,season,purpose,style,usage_conditions
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,shirt dress,women,young adults 18-35,black with white grid,"[button-down, V-neck]",fall,casual,grunge,"[outdoor, casual walk]"
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,sweater,unisex,all ages,heather grey,[ribbed knit],winter,casual,minimalist,[outdoor]
2,images_for_training/Paisley_Rose_Print_Dress_i...,neon_beanie,beanie,unisex,all ages,neon yellow,[],unknown,casual,street style,[unknown]
3,images_for_training/Paisley_Rose_Print_Dress_i...,black_trousers,trousers,unisex,all ages,black,[slim fit],all-season,casual,minimalist,"[outdoor, casual walk]"
4,images_for_training/Paisley_Rose_Print_Dress_i...,black_boots,boots,unisex,all ages,black,[],winter,casual,grunge,[outdoor]


In [102]:
df.describe()

,image_path,item_name,item_type,demographic,age_group,color,design_features,season,purpose,style,usage_conditions
count,62979,62979,62979,62979,62979,62979,62979,62979,62979,62979,62979
unique,15000,363,76,3,3,65,1041,8,28,45,39
top,images_for_training/Socialite_Halter_Dress_img...,plaid_shirt_dress,sweater,unisex,all ages,neon yellow,[ribbed knit],all-season,casual,minimalist,[outdoor]
freq,18,14691,14954,41624,44838,14940,16718,26793,55697,24811,38522


***

#### 3. Кодирование признаков

Нам нужно закодировать все колонки, кроме `image_path` т.к. там лежат пути до фотографий. Для оставшихся колонок действуем следующим образом:
1. `item_type`, `season`, `purpose`, `style` - Скорей всего тут есть значения, которые встречаются 1-2 раза, поэтому можно сгруппировать редкие классы в 'other', а остальные закодировать через LabelEncoder или OHE.
2. `demographic`, `age_group` - Тут можно использовать кодировку OHE сразу т.к. всего 3 уникальных значения
3. `color` - как можно заметить выше, нейросеть писала в json сложные цвета (например, neon yellow или heather grey). Можно попробовать написать простую функцию, которая группирует сложные цвета и приводит их к простым (yellow, grey и т.д.)
4. `design_features`- 1041 уникальное значение. Поскольку здесь лежат списки деталей дизайна, можно попробовать использовать MultiLabelBinarizer. Но чтобы не плодить 1041 колонку, можно выбрать только самые популярные
5. `item_name` - по сути эта колонка дублирует `item_type` и `color`, поэтому она не нужна и ее можно просто удалить.
6. `usage_conditions` - эта колонка тоже дублирует информацию из `purpose` и `style` + по пользовательским фотографиям сложно будет определить, например, одежду для прогулки от одежды для похода в кино, поэтому эта колонка будет беспощадно удалена 

*Поработаем с колонкой `item_type`*

In [103]:
# Поработаем с колонками из пункта 1
df['item_type'].value_counts()[39:76]

item_type
tunic               9
watch               8
tank top            8
shirt               7
wallet              7
wristwatch          7
skirt               5
hoodie              5
glove               4
chain               3
wellies             2
cardigan            2
trench coat         2
welly boots         2
hiking boots        2
sandal              1
sneaker             1
footwear            1
goggles             1
crossbody bag       1
winter boots        1
hair accessory      1
wide-leg pants      1
hairband            1
belt bag            1
sweatshirt          1
sock                1
cross-body bag      1
choker necklace     1
rings               1
headband            1
stretch pants       1
leather belt        1
tote bag            1
wellington boots    1
sleeves             1
glasses             1
Name: count, dtype: int64

In [104]:
df['item_type'].unique()

array(['shirt dress', 'sweater', 'beanie', 'trousers', 'boots', 'socks',
       'shoes', 'jacket', 'hat', 'dress', 'bag', 'scarf', 'accessory',
       'jeans', 'pants', 'boot', 'leggings', 'belt', 'welly boots',
       'shoe', 'ankle boots', 'gloves', 'necklace', 'sneakers', 'joggers',
       'leather jacket', 'choker', 'hosiery', 'handbag', 'jewelry',
       'sunglasses', 'turtleneck', 'wide-leg pants', 'hair accessory',
       'top', 'stockings', 'coat', 'shirt', 't-shirt', 'sock',
       'wristwatch', 'tights', 'eyewear', 'watch', 'skirt', 'tank top',
       'cardigan', 'glove', 'backpack', 'hiking boots', 'sweatshirt',
       'beanies', 'cross-body bag', 'choker necklace', 'wellies',
       'hoodie', 'stretch pants', 'headband', 'rings', 'chain', 'wallet',
       'leather belt', 'wellington boots', 'tote bag', 'sandal',
       'sneaker', 'hairband', 'belt bag', 'goggles', 'footwear',
       'winter boots', 'crossbody bag', 'glasses', 'sleeves', 'tunic',
       'trench coat'], dtype

Колонку `item_type` можно легко сократить в количестве уникальных значений т.к. у нас есть, например, несколько названий 'boots': 'welly boots', 'hiking boots', 'winter boots', 'wellington boots'. Точно такая же логика с остальными словами. А все дополнительные украшения, типа очков, ремней, часов можно сделать 'accessories'

In [105]:
def group_item_type(item):
    item = str(item).lower().strip()
    
    # 1. ПЛАТЬЯ 
    if 'dress' in item or 'tunic' in item:
        return 'dress'
        
    # 2. ОБУВЬ
    if any(word in item for word in ['boot', 'wellies']):
        return 'boots'
    if any(word in item for word in ['shoe', 'sneaker', 'sandal', 'footwear']):
        return 'shoes'
        
    # 3. СУМКИ
    if 'bag' in item or 'wallet' in item or 'handbag' in item:
        return 'bag'
        
    # 4. ШТАНЫ / НИЗ
    if any(word in item for word in ['pants', 'trouser', 'jeans', 'legging', 'jogger', 'tights']):
        return 'pants'
    if 'skirt' in item:
        return 'skirt'
        
    # 5. ВЕРХ (Теплый)
    if any(word in item for word in ['sweater', 'sweatshirt', 'hoodie', 'turtleneck', 'cardigan']):
        return 'sweater'
        
    # 6. ВЕРХ (Легкий)
    if any(word in item for word in ['shirt', 'top', 'blouse']):
        return 'top'
        
    # 7. ВЕРХНЯЯ ОДЕЖДА (Куртки, пальто)
    if 'jacket' in item or 'coat' in item:
        return 'outerwear'
        
    # 8. ГОЛОВНЫЕ УБОРЫ
    if any(word in item for word in ['hat', 'beanie', 'headband', 'hairband']):
        return 'headwear'
        
    # 9. АКСЕССУАРЫ И ПРОЧЕЕ
    if any(word in item for word in ['sock', 'stocking', 'hosiery']):
        return 'socks'
    if any(word in item for word in ['glove', 'scarf', 'sleeves']):
        return 'cold_weather_accessories'
    if any(word in item for word in ['accessory', 'jewelry', 'necklace', 'choker', 'belt', 'ring', 'chain', 'watch', 'glasses', 'goggles', 'eyewear']):
        return 'accessories'
        
    return 'other'

In [106]:
# Применяем функцию к колонке
df['item_type'] = df['item_type'].apply(group_item_type)

print(f"76 значений в колонке 'item_type' сократилось до {df['item_type'].nunique()}")
df['item_type'].value_counts()

76 значений в колонке 'item_type' сократилось до 14


item_type
headwear                    15003
sweater                     14972
dress                       14924
shoes                        5180
outerwear                    4928
pants                        4195
boots                        1692
accessories                  1010
cold_weather_accessories      444
bag                           323
socks                         230
top                            60
other                          13
skirt                           5
Name: count, dtype: int64

Балансировка классов все еще не идеальная, но уже намного лучше. У нас получилось, что в 'skirt' всего 5 значений, поэтому их можно объединить с 'pants' и назвать это 'bottomwear' (одежда для нижней части тела), а в 'top' всего 60 и их можно объединить с 'sweater' и назвать 'upperwea' (одежда для верхней части тела). Оставшиеся 13 шт. 'other' можно просто удалить, чтобы не мешало обучению.

In [107]:
# skirt (5) + pants (4195) = bottomwear (4200)
df['item_type'] = df['item_type'].replace({'skirt': 'bottomwear', 'pants': 'bottomwear'})

# top (60) + sweater (14972) = upperwear (15032)
df['item_type'] = df['item_type'].replace({'top': 'upperwear', 'sweater': 'upperwear'})

df = df[df['item_type'] != 'other'].copy()

In [108]:
df['item_type'].value_counts()

item_type
upperwear                   15032
headwear                    15003
dress                       14924
shoes                        5180
outerwear                    4928
bottomwear                   4200
boots                        1692
accessories                  1010
cold_weather_accessories      444
bag                           323
socks                         230
Name: count, dtype: int64

In [109]:
le_item = LabelEncoder()

df['item_type_encoded'] = le_item.fit_transform(df['item_type'])
item_mapping = dict(zip(le_item.transform(le_item.classes_), le_item.classes_))

print("\nСловарь кодировки (Класс -> Число):")
for num, name in item_mapping.items():
    print(f"{num}: {name}")
    

df = df.drop(['item_type'], axis=1)


Словарь кодировки (Класс -> Число):
0: accessories
1: bag
2: boots
3: bottomwear
4: cold_weather_accessories
5: dress
6: headwear
7: outerwear
8: shoes
9: socks
10: upperwear


In [110]:
df

,image_path,item_name,demographic,age_group,color,design_features,season,purpose,style,usage_conditions,item_type_encoded
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,women,young adults 18-35,black with white grid,"[button-down, V-neck]",fall,casual,grunge,"[outdoor, casual walk]",5
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,unisex,all ages,heather grey,[ribbed knit],winter,casual,minimalist,[outdoor],10
2,images_for_training/Paisley_Rose_Print_Dress_i...,neon_beanie,unisex,all ages,neon yellow,[],unknown,casual,street style,[unknown],6
3,images_for_training/Paisley_Rose_Print_Dress_i...,black_trousers,unisex,all ages,black,[slim fit],all-season,casual,minimalist,"[outdoor, casual walk]",3
4,images_for_training/Paisley_Rose_Print_Dress_i...,black_boots,unisex,all ages,black,[],winter,casual,grunge,[outdoor],2
...,...,...,...,...,...,...,...,...,...,...,...
62974,images_for_training/WOMEN1_Blouses_Shirts_id_0...,head,unisex,all ages,neon yellow,[ribbed knit],winter,casual,grunge,[outdoor],6
62975,images_for_training/Striped_Ringer_Pocket_Tee_...,outer_layer,women,young adults 18-35,black with white grid,"[button-down, V-neck]",all-season,casual,minimalist,[outdoor],5
62976,images_for_training/Striped_Ringer_Pocket_Tee_...,footwear,women,young adults 18-35,black,"[lacing, treaded sole]",winter,casual,grunge,[outdoor],2
62977,images_for_training/Striped_Ringer_Pocket_Tee_...,top_layer,women,young adults 18-35,heather grey,[ribbed knit],winter,casual,grunge,[indoor],10


***

*Поработаем с колонкой `season`*

In [111]:
df['season'].value_counts()

season
all-season      26780
winter          20507
fall            15542
summer             52
spring             40
unknown            30
fall, winter        8
winter, fall        7
Name: count, dtype: int64

Тут можно объединить `fall, winter` с `fall`, `winter, fall` с `winter`, `spring` с `summer` и назвать `spring_summer`, а `unknown` вовсе удалить

In [112]:
# fall, winter (8) + fall (15542) = fall (15550)
df['season'] = df['season'].replace({'fall, winter': 'fall', 'fall': 'fall'})

# winter, fall (7) + winter (20507) = winter (20514)
df['season'] = df['season'].replace({'winter, fall': 'winter', 'winter': 'winter'})

# spring (40) + summer (52) = spring_summer (92)
df['season'] = df['season'].replace({'spring': 'spring_summer', 'summer': 'spring_summer'})

df = df[df['season'] != 'unknown'].copy()

In [113]:
df['season'].value_counts()

season
all-season       26780
winter           20514
fall             15550
spring_summer       92
Name: count, dtype: int64

*Кодировка `season` с помощью OHE*

In [114]:
df = pd.get_dummies(df, columns=['season'], prefix='is_season', dtype=int)

In [115]:
df.head(2)

,image_path,item_name,demographic,age_group,color,design_features,purpose,style,usage_conditions,item_type_encoded,is_season_all-season,is_season_fall,is_season_spring_summer,is_season_winter
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,women,young adults 18-35,black with white grid,"[button-down, V-neck]",casual,grunge,"[outdoor, casual walk]",5,0,1,0,0
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,unisex,all ages,heather grey,[ribbed knit],casual,minimalist,[outdoor],10,0,0,0,1


***

*Поработаем с колонкой `purpose`*

In [116]:
df['purpose'].value_counts()

purpose
casual                  55673
street style             5474
accessory                1192
footwear                  183
outdoor                   176
shoes                     100
outerwear                  50
fashion                    27
fashion statement          13
functional                 13
formal                      6
outer layer                 5
undergarment                3
activewear                  3
practical                   2
jewelry                     2
bag                         2
daily use                   2
jacket                      2
fashion accessory           1
active wear                 1
shoe                        1
casual walk                 1
dressy                      1
cold weather                1
casual, street style        1
athleisure                  1
Name: count, dtype: int64

Тут есть значения, которые не описывают цель ношения (`shoes`, `outerwear`, `accessory`, `bag` и тд) и их достаточнор много, поэтому лучше всего будет не удалять их, а переместить в `other`. Также можно объединить синонимы, как это делалось выше и после этого уже можно посмотреть, как лучше всего кодировать

In [117]:
def group_purpose(p):
    p = str(p).lower().strip()
    
    # 1. Повседневная и уличная одежда
    if any(word in p for word in ['casual', 'street', 'daily']):
        return 'casual'
        
    # 2. Спорт и активность
    if any(word in p for word in ['active', 'athleisure']):
        return 'sport'
        
    # 3. Мода и нарядная одежда
    if 'fashion' in p or 'dressy' in p or 'formal' in p:
        return 'formal_fashion'
        
    # 4. Функциональная/Уличная одежда 
    if any(word in p for word in ['functional', 'practical', 'outdoor', 'cold']):
        return 'outdoor'
        
    # Все остальное 
    return 'other'

In [118]:
df['purpose'] = df['purpose'].apply(group_purpose)

In [119]:
df['purpose'].value_counts()

purpose
casual            61151
other              1540
outdoor             192
formal_fashion       48
sport                 5
Name: count, dtype: int64

Можно было бы тут еще объединить некоторые колонки, но набор данных с повседневной одеждой все-равно будет больше всех из-за чего колонка `purpose` не несет в себе никакой полезной информации, поэтому она будет просто удалена

In [120]:
df = df.drop(['purpose'], axis=1)
df.head(2)

,image_path,item_name,demographic,age_group,color,design_features,style,usage_conditions,item_type_encoded,is_season_all-season,is_season_fall,is_season_spring_summer,is_season_winter
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,women,young adults 18-35,black with white grid,"[button-down, V-neck]",grunge,"[outdoor, casual walk]",5,0,1,0,0
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,unisex,all ages,heather grey,[ribbed knit],minimalist,[outdoor],10,0,0,0,1


***

*Поработаем с колонкой `style`*

In [121]:
df['style'].value_counts()

style
minimalist           24801
grunge               19596
street style         12738
casual                2317
bohemian              1481
boho chic              568
streetwear             433
boho                   324
outdoor                221
edgy                   137
athleisure             127
fashion                 66
rustic                  18
biker                   16
classic                 16
practical                9
fashion statement        8
winter                   6
rock                     5
winter sports            5
winter style             4
functional               4
leather jacket           4
business casual          3
leather                  3
motorcycle               2
unknown                  2
fun                      2
boots                    2
utilitarian              2
winter wear              2
rock and roll            1
outdoor style            1
industrial               1
fashionable              1
goth                     1
neon                  

Тут можно сделать следующие группировки:
* `minimalist`, `classic`, `business casual`: Чистые линии, базовые вещи, классика -> `minimalist_classic`
* `grunge`, `edgy`, `rock`, `biker`, `goth`: Дерзкий, темный стиль, кожа, металл -> `grunge_edgy`
* `street style`, `streetwear`, `casual`: Уличная и повседневная мода -> `streetwear_casual`
* `bohemian`, `boho chic`, `boho`, `rustic`: Легкие ткани, принты, свободный крой -> `boho`
* `athleisure`, `outdoor`, `functional`, `winter sports`: Спорт и активный отдых -> `sport_outdoor`
* Все остальное сначала объединю в `other` и потом удалю, если будет мало значений

In [122]:
def group_style(s):
    s = str(s).lower().strip()
    
    # 1. дерзкий стиль
    if any(word in s for word in ['grunge', 'edgy', 'rock', 'biker', 'goth', 'motorcycle']):
        return 'grunge_edgy'
        
    # 2. Уличный и кэжуал
    if any(word in s for word in ['street', 'casual']):
        return 'streetwear_casual'
        
    # 3. Бохо
    if 'boho' in s or 'bohemian' in s or 'rustic' in s:
        return 'boho'
        
    # 4. Спорт и Аутдор
    if any(word in s for word in ['athleisure', 'outdoor', 'functional', 'practical', 'utilitarian', 'sports']):
        return 'sport_outdoor'
        
    # 5. Минимализм и Классика
    if any(word in s for word in ['minimalist', 'classic', 'preppy']):
        return 'minimalist_classic'
        
    return 'other'

In [123]:
df['style'] = df['style'].apply(group_style)

In [124]:
df['style'].value_counts()

style
minimalist_classic    24818
grunge_edgy           19758
streetwear_casual     15492
boho                   2391
sport_outdoor           369
other                   108
Name: count, dtype: int64

`other` можно удалить и прменить после OHE кодировку

In [125]:
df = df[df['style'] != 'other'].copy()

In [126]:
df['style'].value_counts()

style
minimalist_classic    24818
grunge_edgy           19758
streetwear_casual     15492
boho                   2391
sport_outdoor           369
Name: count, dtype: int64

In [127]:
df = pd.get_dummies(df, columns=['style'], prefix='is_style', dtype=int)

In [128]:
df.head(2)

,image_path,item_name,demographic,age_group,color,design_features,usage_conditions,item_type_encoded,is_season_all-season,is_season_fall,is_season_spring_summer,is_season_winter,is_style_boho,is_style_grunge_edgy,is_style_minimalist_classic,is_style_sport_outdoor,is_style_streetwear_casual
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,women,young adults 18-35,black with white grid,"[button-down, V-neck]","[outdoor, casual walk]",5,0,1,0,0,0,1,0,0,0
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,unisex,all ages,heather grey,[ribbed knit],[outdoor],10,0,0,0,1,0,0,1,0,0


***

*Поработаем с колонками `demographic`, `age_group`*

In [129]:
df['demographic'].value_counts()

demographic
unisex    41476
women     16489
men        4863
Name: count, dtype: int64

In [130]:
df['age_group'].value_counts()

age_group
all ages              44690
young adults 18-35    18130
teenagers                 8
Name: count, dtype: int64

Тут можно сразу применять OHE кодировку. Единственное, я бы только для колонки `age_group` объединила группы `young adults 18-35` и `teenagers` в `young adults under 35`

In [131]:
# teenagers (8) + young adults 18-35 (18130) = young adults under 35 (18138)
df['age_group'] = df['age_group'].replace({'teenagers': 'young adults under 35', 'young adults 18-35': 'young adults under 35'})

In [132]:
df['age_group'].value_counts()

age_group
all ages                 44690
young adults under 35    18138
Name: count, dtype: int64

In [133]:
df = pd.get_dummies(df, columns=['demographic'], prefix='is_demographic', dtype=int)
df = pd.get_dummies(df, columns=['age_group'], prefix='is_age_group', dtype=int)

In [134]:
df.head(2)

,image_path,item_name,color,design_features,usage_conditions,item_type_encoded,is_season_all-season,is_season_fall,is_season_spring_summer,is_season_winter,is_style_boho,is_style_grunge_edgy,is_style_minimalist_classic,is_style_sport_outdoor,is_style_streetwear_casual,is_demographic_men,is_demographic_unisex,is_demographic_women,is_age_group_all ages,is_age_group_young adults under 35
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,black with white grid,"[button-down, V-neck]","[outdoor, casual walk]",5,0,1,0,0,0,1,0,0,0,0,0,1,0,1
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,heather grey,[ribbed knit],[outdoor],10,0,0,0,1,0,0,1,0,0,0,1,0,1,0


***

*Поработаем с колонкой `color`*

In [135]:
df['color'].value_counts()

color
heather grey                           14939
neon yellow                            14847
black                                  14514
black with white grid                  13044
black with white plaid                  1922
                                       ...  
green                                      1
grey and white striped                     1
blue with white stitching                  1
dark brown with light brown stripes        1
white with grey stripes                    1
Name: count, Length: 65, dtype: int64

Как выше и писала, нужно привести цвета к стандартным перед тем, как делать кодировку. Напишем функцию

In [136]:
def simplify_color(c):
    c = str(c).lower().strip()
    
    # 1. Принты, узоры и мультиколор 
    if any(word in c for word in ['multi', 'floral', 'plaid', 'grid', 'stripe', 'pattern', 'leopard']):
        return 'multicolor_print'
        
    # 2. Черно-белая классика
    if 'black' in c and 'white' in c:
        return 'black_and_white'
        
    # 3. Базовые моно-цвета
    if 'black' in c: return 'black'
    if 'white' in c: return 'white'
    
    if 'grey' in c or 'gray' in c or 'silver' in c: return 'grey'
    if 'red' in c or 'burgundy' in c or 'maroon' in c or 'wine' in c: return 'red'
    if 'blue' in c or 'navy' in c or 'denim' in c or 'teal' in c: return 'blue'
    if 'green' in c or 'olive' in c or 'khaki' in c or 'mint' in c: return 'green'
    
    if 'yellow' in c or 'mustard' in c or 'gold' in c: return 'yellow'
    if 'pink' in c or 'fuchsia' in c or 'rose' in c: return 'pink'
    if 'purple' in c or 'violet' in c or 'lavender' in c: return 'purple'
    if 'orange' in c or 'peach' in c or 'coral' in c: return 'orange'
    
    # 4. Коричневые и бежевые оттенки
    if 'brown' in c or 'camel' in c or 'tan' in c or 'rust' in c: return 'brown'
    if 'beige' in c or 'cream' in c or 'ivory' in c or 'nude' in c: return 'beige'
    
    return 'other'

In [137]:
df['color'] = df['color'].apply(simplify_color)

Теперь удалим данные, где не получилось определить цвет

In [139]:
df = df[df['color'] != 'other'].copy()

In [141]:
df['color'].value_counts()

color
multicolor_print    15112
grey                15110
yellow              14850
black               14522
brown                1491
white                 832
blue                  678
black_and_white        22
green                  22
red                    19
beige                   6
pink                    5
Name: count, dtype: int64

In [142]:
df = pd.get_dummies(df, columns=['color'], prefix='is_color', dtype=int)

In [143]:
df.head(2)

,image_path,item_name,design_features,usage_conditions,item_type_encoded,is_season_all-season,is_season_fall,is_season_spring_summer,is_season_winter,is_style_boho,...,is_color_black_and_white,is_color_blue,is_color_brown,is_color_green,is_color_grey,is_color_multicolor_print,is_color_pink,is_color_red,is_color_white,is_color_yellow
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,"[button-down, V-neck]","[outdoor, casual walk]",5,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,[ribbed knit],[outdoor],10,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0


***

*Поработаем с колонками `item_name`, `usage_conditions`*

In [144]:
df = df.drop(['item_name', 'usage_conditions'], axis=1)

In [145]:
df.shape

(62669, 29)

***

*Поработаем с колонкой `design_features`*

In [146]:
df['design_features'].value_counts()

design_features
[ribbed knit]              16717
[]                         12466
[button-down, V-neck]       9637
[button-down]               3475
[slouchy fit]               1242
                           ...  
[parka style, fur trim]        1
[ankle strap, lace up]         1
[wool, knit]                   1
[leather, padded]              1
[flared]                       1
Name: count, Length: 1035, dtype: int64

Возьмем только топ-15 деталей, а остальное удалим и после закодируем

In [148]:
all_features = df['design_features'].explode()
all_features = all_features.dropna()
feature_counts = all_features.value_counts()

print("Топ-15 самых частых деталей:")
print(feature_counts.head(15))

Топ-15 самых частых деталей:
design_features
ribbed knit          18370
button-down          14089
V-neck               11519
rubber sole           1553
slouchy fit           1315
zipped pockets        1262
lace-up               1206
button-front          1000
pockets                977
zipped                 971
button-fly             808
elastic waistband      689
biker jacket           636
crew-neck              621
ankle-high             602
Name: count, dtype: int64


In [149]:
top_features_set = set(feature_counts.head(15).index)

df['design_features'] = df['design_features'].apply(
    lambda lst: [item for item in lst if item in top_features_set] if isinstance(lst, list) else []
)

In [151]:
# Применяем MultiLabelBinarizer
mlb = MultiLabelBinarizer()
features_encoded = mlb.fit_transform(df['design_features'])

feature_columns = [f"feature_{c.replace(' ', '_')}" for c in mlb.classes_]
df_features = pd.DataFrame(features_encoded, columns=feature_columns, index=df.index)

df = pd.concat([df, df_features], axis=1)
df = df.drop(['design_features'], axis=1)

In [152]:
df

,image_path,item_type_encoded,is_season_all-season,is_season_fall,is_season_spring_summer,is_season_winter,is_style_boho,is_style_grunge_edgy,is_style_minimalist_classic,is_style_sport_outdoor,...,feature_button-front,feature_crew-neck,feature_elastic_waistband,feature_lace-up,feature_pockets,feature_ribbed_knit,feature_rubber_sole,feature_slouchy_fit,feature_zipped,feature_zipped_pockets
0,images_for_training/Paisley_Rose_Print_Dress_i...,5,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,images_for_training/Paisley_Rose_Print_Dress_i...,10,0,0,0,1,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3,images_for_training/Paisley_Rose_Print_Dress_i...,3,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,images_for_training/Paisley_Rose_Print_Dress_i...,2,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,images_for_training/Paisley_Rose_Print_Dress_i...,9,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62974,images_for_training/WOMEN1_Blouses_Shirts_id_0...,6,0,0,0,1,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
62975,images_for_training/Striped_Ringer_Pocket_Tee_...,5,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
62976,images_for_training/Striped_Ringer_Pocket_Tee_...,2,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
62977,images_for_training/Striped_Ringer_Pocket_Tee_...,10,0,0,0,1,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0


#### 4. Сохранение датасета для ML

In [153]:
Path('data').mkdir(exist_ok=True)

df.to_csv('data/processed_dataset.csv', index=False)

print(f"Итоговый размер таблицы: {df.shape}")

Итоговый размер таблицы: (62669, 43)


**[Скачать датасет и аннотации (Google Drive)](https://drive.google.com/drive/folders/1NaYi4rhdZP9EKPB4gG2ryciebfsMau_3?usp=sharing)**